# Study 955 — ADR Catch-Up — the teardown

The stale-price HAC regression, the index-leg/FX-leg split, the region cut by closing time, the trim/winsorize knife that decides whether the one surviving coefficient is a loading or a tail, `b1` versus the bettable γ and the autocorrelation that separates them, the Fama-MacBeth residual test and its bounce confound, the costed books with breakeven costs, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `a564d6b737e5`), sample 2004-01-05 → 2026-06-30, 41272 name-days, one execution lag. ADR closes total-return; home indices price-only.

In [1]:
R = {'start': '2004-01-05', 'end': '2026-06-30', 'n_rows': 41272, 'n_names': 8, 'fp': 'a564d6b737e5', 'alpha_bps': 3.63, 't_alpha': 3.22, 'b0': 0.5761, 't_b0': 25.0, 'b1': 0.0209, 't_b1': 1.15, 'n_obs': 41264, 'all_h': 0.68, 'all_t_h': 32.3, 'all_hlag': 0.0023, 'all_t_hlag': 0.12, 'all_f': 0.235, 'all_t_f': 6.9, 'all_flag': 0.0683, 'all_t_flag': 2.79, 'jp_h': 0.362, 'jp_t_h': 14.6, 'jp_hlag': 0.062, 'jp_t_hlag': 2.43, 'jp_f': -0.081, 'jp_t_f': -1.5, 'jp_flag': 0.0717, 'jp_t_flag': 1.99, 'uk_hlag': -0.0421, 'uk_t_hlag': -1.31, 'eu_hlag': -0.0208, 'eu_t_hlag': -0.87, 'jp_n': 9650, 'jp_b0': 0.29, 'jp_b1': 0.0809, 'jp_t_b1': 3.25, 'jp_gamma': 0.0331, 'jp_t_gamma': 1.43, 'jp_rho': -0.165, 'eu_n': 10558, 'eu_b0': 0.471, 'eu_b1': 0.0123, 'eu_t_b1': 0.47, 'eu_gamma': -0.0051, 'eu_t_gamma': -0.24, 'uk_n': 21056, 'uk_b0': 0.843, 'uk_b1': -0.0479, 'uk_t_b1': -1.8, 'uk_gamma': -0.0357, 'uk_t_gamma': -1.16, 'tm_b1': 0.0936, 'tm_t_b1': 3.08, 'sony_b1': 0.0681, 'sony_t_b1': 2.75, 'all_gamma': -0.0077, 'all_t_gamma': -0.41, 'all_rho': -0.05, 'jp_blocks': [('2004-2009', 2550, 0.0806, 1.74, 0.0347, 0.79), ('2010-2014', 2160, 0.1119, 2.35, 0.0551, 1.1), ('2015-2020', 2590, 0.054, 1.15, -0.0012, -0.03), ('2021-2026', 2344, 0.0658, 2.1, 0.0391, 1.24)], 'jp_early_b1': 0.0935, 'jp_early_t': 2.6, 'jp_late_b1': 0.0598, 'jp_late_t': 2.25, 'jp_tail': [('full sample', 9650, 0.0809, 3.25), ('trim >99.5th pct', 9602, 0.0169, 0.67), ('trim >99th pct', 9554, 0.0307, 1.52), ('trim >97.5th pct', 9408, 0.0358, 1.75), ('trim >95th pct', 9168, 0.0418, 1.82), ('winsorize @99th', 9650, 0.0603, 2.62), ('winsorize @97.5th', 9650, 0.0567, 2.58)], 'syn_tail': [('full sample', 0.251, 47.8), ('trim >99th pct', 0.2489, 45.4), ('trim >95th pct', 0.2511, 40.4)], 'jp_buckets': [(1, -0.0192, 0.0), (2, -0.0053, 4.81), (3, 0.0005, 6.38), (4, 0.0061, 1.23), (5, 0.0191, 1.6)], 'jp_q5q1': 1.6, 'jp_consec_share': 0.92, 'jp_consec_b1': 0.0749, 'jp_consec_t': 3.29, 'jp_boot_lo': 0.0338, 'jp_boot_hi': 0.1225, 'jp_boot_neg': 0.0, 'fm_rank_bp': 4.6, 'fm_rank_t': 1.32, 'fm_raw': 0.0214, 'fm_raw_t': 2.22, 'pool_e': -0.0168, 'pool_e_t': -1.06, 'coef_a': -0.023, 't_a': -1.57, 'coef_x': 0.0056, 't_x': 0.27, 'cu_gross': -0.92, 'cu_sharpe': -0.049, 'cu_t': -0.23, 'cu_net': -14.81, 'cu_be': -0.34, 're_gross': 2.55, 're_sharpe': 0.161, 're_t': 0.72, 're_net': -12.01, 're_be': 0.89, 'rv_gross': 5.28, 'rv_sharpe': 0.326, 'rv_t': 1.66, 'rv_net': -8.57, 'rv_be': 1.94, 'jp_cu_gross': 2.3, 'jp_cu_sharpe': 0.09, 'jp_cu_t': 0.41, 'jp_cu_net': -11.52, 'jp_cu_be': 0.85, 'turnover': 1.083, 'ci_cu_lo': -0.469, 'ci_cu_hi': 0.396, 'ci_cu_neg': 57.4, 'ci_re_lo': -0.29, 'ci_re_hi': 0.6, 'ci_rv_lo': -0.08, 'ci_rv_hi': 0.706, 'cost_grid': [(0.0, -1.16, -0.062), (1.0, -3.89, -0.206), (2.0, -6.62, -0.351), (5.0, -14.81, -0.785), (10.0, -28.46, -1.506), (25.0, -69.42, -3.641)], 'borrow_lo': -14.57, 'borrow_hi': -16.01, 'cu_dn_sharpe': -0.369, 'cu_lin_sharpe': 0.01, 'era_e_n': 20148, 'era_e_b1': 0.0285, 'era_e_t': 1.12, 'era_e_gamma': -0.014, 'era_l_n': 21108, 'era_l_b1': 0.0103, 'era_l_t': 0.44, 'era_l_gamma': 0.0018, 'nvo_real_b1': 0.0194, 'nvo_real_t': 0.43, 'nvo_proxy_b1': 0.0381, 'nvo_proxy_t': 0.78, 'ew_ann': 11.19, 'ew_sharpe': 0.515, 'ew_t': 2.67, 'ew_bil_sharpe': 0.45, 'ew_bil_t': 2.15, 'syn_b1': 0.251, 'syn_t_b1': 47.8, 'syn_sharpe': 7.54, 'syn_null_b1': 0.0002, 'syn_null_sd': 0.004, 'syn_null_fire': 0, 'syn_bounce_fm': -0.1535, 'syn_bounce_fm_t': -20.6, 'syn_bounce_b1': 0.006, 'syn_bounce_t': 0.96, 'syn_bounce_xa': 0.1086, 'syn_bounce_xa_t': 13.3}

> 💡 **In plain words.** `b1` is the answer to: *knowing what the home market did today, does what it did yesterday still tell me anything about the ADR today?* If the ADR closed at fair value, the answer is no.

## 1. The pooled stale-price regression

`a_t = α + b0·x_t + b1·x_{t-1}`, HAC (Newey-West) standard errors, rows pooled across names and sorted by date so the Bartlett kernel absorbs both serial and same-day cross-sectional correlation. `b0` is the stock's beta to its home *index*, not a catch-up ratio — it has no reason to be 1.

In [2]:
print(f"alpha {R['alpha_bps']:+.2f} bp/day (t={R['t_alpha']:+.2f})")
print(f"b0  same-day home dollar move : {R['b0']:+.4f}  (t={R['t_b0']:+.1f})")
print(f"b1  LAGGED home dollar move   : {R['b1']:+.4f}  (t={R['t_b1']:+.2f})   <- the test")
print(f"n = {R['n_obs']:,}")

alpha +3.63 bp/day (t=+3.22)
b0  same-day home dollar move : +0.5761  (t=+25.0)
b1  LAGGED home dollar move   : +0.0209  (t=+1.15)   <- the test
n = 41,264


## 2. Where does the pooled lag actually live — the index or the currency?

`a_t = α + c0·h_t + c1·h_{t-1} + d0·f_t + d1·f_{t-1}`. A stale-*market* story has to show up in the index leg. FX trades around the clock and Yahoo stamps its close in New York hours, so a lagged FX loading is a snapshot artefact, not unpaid information.

In [3]:
print(f"ALL   h_lag {R['all_hlag']:+.4f} (t={R['all_t_hlag']:+.2f})   "
      f"f_lag {R['all_flag']:+.4f} (t={R['all_t_flag']:+.2f})  <- the pooled lag is ALL currency")
print(f"Japan h_lag {R['jp_hlag']:+.4f} (t={R['jp_t_hlag']:+.2f})   "
      f"f_lag {R['jp_flag']:+.4f} (t={R['jp_t_flag']:+.2f})  <- a genuine stale INDEX loading")
print(f"UK    h_lag {R['uk_hlag']:+.4f} (t={R['uk_t_hlag']:+.2f})")
print(f"EU    h_lag {R['eu_hlag']:+.4f} (t={R['eu_t_hlag']:+.2f})")

ALL   h_lag +0.0023 (t=+0.12)   f_lag +0.0683 (t=+2.79)  <- the pooled lag is ALL currency
Japan h_lag +0.0620 (t=+2.43)   f_lag +0.0717 (t=+1.99)  <- a genuine stale INDEX loading
UK    h_lag -0.0421 (t=-1.31)
EU    h_lag -0.0208 (t=-0.87)


## 3. The region cut — the discriminating test

Tokyo closes 02:00 ET, thirteen hours before the ADR. London and Frankfurt close 11:30 ET, *inside* the US session. This is not a subgroup hunt: the closing clock is the mechanism, so the cut is the hypothesis, and the UK null is a confirmation rather than a failure.

In [4]:
for tag, n, b0, b1, tb1, g, tg in [
    ('Japan ', R['jp_n'], R['jp_b0'], R['jp_b1'], R['jp_t_b1'], R['jp_gamma'], R['jp_t_gamma']),
    ('Europe', R['eu_n'], R['eu_b0'], R['eu_b1'], R['eu_t_b1'], R['eu_gamma'], R['eu_t_gamma']),
    ('UK    ', R['uk_n'], R['uk_b0'], R['uk_b1'], R['uk_t_b1'], R['uk_gamma'], R['uk_t_gamma'])]:
    print(f'{tag} n={n:6,}  b0={b0:+.3f}  b1={b1:+.4f} (t={tb1:+.2f})  gamma={g:+.4f} (t={tg:+.2f})')
print()
print(f"per name: TM b1={R['tm_b1']:+.4f} (t={R['tm_t_b1']:+.2f}), "
      f"SONY b1={R['sony_b1']:+.4f} (t={R['sony_t_b1']:+.2f})")

Japan  n= 9,650  b0=+0.290  b1=+0.0809 (t=+3.25)  gamma=+0.0331 (t=+1.43)
Europe n=10,558  b0=+0.471  b1=+0.0123 (t=+0.47)  gamma=-0.0051 (t=-0.24)
UK     n=21,056  b0=+0.843  b1=-0.0479 (t=-1.80)  gamma=-0.0357 (t=-1.16)

per name: TM b1=+0.0936 (t=+3.08), SONY b1=+0.0681 (t=+2.75)


### Japan block by block

In [5]:
for tag, n, b1, tb1, g, tg in R['jp_blocks']:
    print(f'{tag} n={n:5d}: b1={b1:+.4f} (t={tb1:+.2f})   gamma={g:+.4f} (t={tg:+.2f})')
print(f"2004-2014 half: b1={R['jp_early_b1']:+.4f} (t={R['jp_early_t']:+.2f})")
print(f"2015-2026 half: b1={R['jp_late_b1']:+.4f} (t={R['jp_late_t']:+.2f})")

2004-2009 n= 2550: b1=+0.0806 (t=+1.74)   gamma=+0.0347 (t=+0.79)
2010-2014 n= 2160: b1=+0.1119 (t=+2.35)   gamma=+0.0551 (t=+1.10)
2015-2020 n= 2590: b1=+0.0540 (t=+1.15)   gamma=-0.0012 (t=-0.03)
2021-2026 n= 2344: b1=+0.0658 (t=+2.10)   gamma=+0.0391 (t=+1.24)
2004-2014 half: b1=+0.0935 (t=+2.60)
2015-2026 half: b1=+0.0598 (t=+2.25)


## 3½. Is that Japan loading linear, or does the tail own it?

The question the HAC *t* cannot answer. A HAC standard error corrects for dependence, not for *leverage*: a slope is an average, and a handful of enormous regressor values can be its sole author. Under linearity, selecting on a regressor does not bias OLS — so if the loading is real and uniform, trimming the biggest `|x_lag|` rows should barely move it, and winsorizing (capping, keeping every row) should move it not at all. Both knives, plus the calibration on a synthetic panel where the planted lag *is* linear:

In [6]:
print('Japan:')
for tag, n, b1, t in R['jp_tail']:
    print(f'  {tag:20s} n={n:6,}  b1={b1:+.4f} (t={t:+.2f})')
print()
print('SYNTHETIC calibration - a genuinely LINEAR planted lag under the same knife:')
for tag, b1, t in R['syn_tail']:
    print(f'  {tag:20s}          b1={b1:+.4f} (t={t:+.1f})')
drop = 1 - R['jp_tail'][1][2] / R['jp_tail'][0][2]
syn_drop = 1 - R['syn_tail'][1][1] / R['syn_tail'][0][1]
print()
print(f'0.5% of rows removed: Japan loses {drop:.0%} of its coefficient, '
      f'the linear plant loses {syn_drop:.0%} at a 1% trim.')

Japan:
  full sample          n= 9,650  b1=+0.0809 (t=+3.25)
  trim >99.5th pct     n= 9,602  b1=+0.0169 (t=+0.67)
  trim >99th pct       n= 9,554  b1=+0.0307 (t=+1.52)
  trim >97.5th pct     n= 9,408  b1=+0.0358 (t=+1.75)
  trim >95th pct       n= 9,168  b1=+0.0418 (t=+1.82)
  winsorize @99th      n= 9,650  b1=+0.0603 (t=+2.62)
  winsorize @97.5th    n= 9,650  b1=+0.0567 (t=+2.58)

SYNTHETIC calibration - a genuinely LINEAR planted lag under the same knife:
  full sample                   b1=+0.2510 (t=+47.8)
  trim >99th pct                b1=+0.2489 (t=+45.4)
  trim >95th pct                b1=+0.2511 (t=+40.4)

0.5% of rows removed: Japan loses 79% of its coefficient, the linear plant loses 1% at a 1% trim.


### The non-parametric twin: sort by yesterday's home move

No regression, no leverage: bucket every Japanese name-day by `x_lag` and read what the ADR paid next. (Rows inside a bucket share dates and home markets, so a *t* here would overstate the evidence — read the basis points.)

In [7]:
for b, x, a in R['jp_buckets']:
    print(f'  Q{b}  x_lag {x:+.4f}  ->  ADR next {a:+6.2f} bp')
print(f"\n  Q5 - Q1 = {R['jp_q5q1']:+.2f} bp, and not monotone.")
print()
print(f"lag is strictly yesterday ({R['jp_consec_share']:.1%} of rows): "
      f"b1={R['jp_consec_b1']:+.4f} (t={R['jp_consec_t']:+.2f}) - unchanged")
print(f"block bootstrap over dates: b1 95% CI "
      f"[{R['jp_boot_lo']:+.4f}, {R['jp_boot_hi']:+.4f}], "
      f"{R['jp_boot_neg']:.1f}% of draws <= 0")
print()
print('The bootstrap says it is not a sampling fluke; the trim says it is a')
print('body-vs-tail fluke. Those are different objections and both are reported.')

  Q1  x_lag -0.0192  ->  ADR next  +0.00 bp
  Q2  x_lag -0.0053  ->  ADR next  +4.81 bp
  Q3  x_lag +0.0005  ->  ADR next  +6.38 bp
  Q4  x_lag +0.0061  ->  ADR next  +1.23 bp
  Q5  x_lag +0.0191  ->  ADR next  +1.60 bp

  Q5 - Q1 = +1.60 bp, and not monotone.

lag is strictly yesterday (92.0% of rows): b1=+0.0749 (t=+3.29) - unchanged
block bootstrap over dates: b1 95% CI [+0.0338, +0.1225], 0.0% of draws <= 0

The bootstrap says it is not a sampling fluke; the trim says it is a
body-vs-tail fluke. Those are different objections and both are reported.


> 💡 **In plain words.** Forty-eight nights out of nine thousand six hundred and fifty carry the whole result. That is not a reason to call it fake — something does happen after a huge Tokyo session — but it is a decisive reason not to call it an established loading, and it is why the Signal stamp on this study is Weak and not Mixed.

## 4. `b1` is not the tradable coefficient — and the gap is arithmetic

`b1` conditions on `x_{t+1}`, which is unknown at the trade. The bettable coefficient is the univariate γ in `a_{t+1} = α + γ·x_t`. Because the home dollar move is itself negatively autocorrelated, the two-regressor estimator inflates `b1` — part of the lagged loading is the estimator undoing the same-day loading applied to a mean-reverting regressor.

In [8]:
print(f"ALL   b1={R['b1']:+.4f} (t={R['t_b1']:+.2f})  ->  gamma={R['all_gamma']:+.4f} "
      f"(t={R['all_t_gamma']:+.2f})   rho1(x)={R['all_rho']:+.3f}")
print(f"Japan b1={R['jp_b1']:+.4f} (t={R['jp_t_b1']:+.2f})  ->  gamma={R['jp_gamma']:+.4f} "
      f"(t={R['jp_t_gamma']:+.2f})   rho1(x)={R['jp_rho']:+.3f}")
print(f"\ninflation factor in Japan: {R['jp_b1']/R['jp_gamma']:.2f}x")
print('gamma clears |t| = 2 nowhere.')

ALL   b1=+0.0209 (t=+1.15)  ->  gamma=-0.0077 (t=-0.41)   rho1(x)=-0.050
Japan b1=+0.0809 (t=+3.25)  ->  gamma=+0.0331 (t=+1.43)   rho1(x)=-0.165

inflation factor in Japan: 2.44x
gamma clears |t| = 2 nowhere.


> 💡 **In plain words.** Quoting the big coefficient would have made the prize look about two and a half times larger than anything you could actually bet on. That single substitution is the difference between a headline and an honest result.

## 5. The residual test, and why it carries no weight

`e_t = a_t − β_t·x_t`, with `β_t` a 252-day rolling beta on data through `t-1` only. Three ways of testing whether `e_t` predicts `a_{t+1}` disagree *in sign* — which is itself the finding. The raw cross-sectional slope divides by the day's dispersion, near zero on a quiet eight-name cross-section; the pooled slope ignores cross-correlation and overstates its own *t*. The rank-standardised Fama-MacBeth is the one to read.

In [9]:
print(f"Fama-MacBeth, rank-standardised : {R['fm_rank_bp']:+.2f} bp  (t={R['fm_rank_t']:+.2f})  <- read this one")
print(f"Fama-MacBeth, raw slope         : {R['fm_raw']:+.4f}      (t={R['fm_raw_t']:+.2f})  <- unstable")
print(f"pooled slope                    : {R['pool_e']:+.4f}     (t={R['pool_e_t']:+.2f})  <- over-optimistic")

Fama-MacBeth, rank-standardised : +4.60 bp  (t=+1.32)  <- read this one
Fama-MacBeth, raw slope         : +0.0214      (t=+2.22)  <- unstable
pooled slope                    : -0.0168     (t=-1.06)  <- over-optimistic


### And the discriminating regression: `a_{t+1} = α + φ·a_t + γ·x_t`

Any residual contains `a_t`, so a negative slope on it may just be Roll bounce and Nagel liquidity-provision reversal. Put the ADR's own move in the regression and ask what the home tape adds.

In [10]:
print(f"own move a_t : {R['coef_a']:+.4f}  (t={R['t_a']:+.2f})   <- plain one-day reversal")
print(f"home    x_t  : {R['coef_x']:+.4f}  (t={R['t_x']:+.2f})   <- the home tape adds nothing")
print()
print('And this test is BIASED IN FAVOUR of catch-up: on a synthetic panel with')
print(f"pure bounce and ZERO stale information it returns {R['syn_bounce_xa']:+.4f} "
      f"(t={R['syn_bounce_xa_t']:+.1f}).")

own move a_t : -0.0230  (t=-1.57)   <- plain one-day reversal
home    x_t  : +0.0056  (t=+0.27)   <- the home tape adds nothing

And this test is BIASED IN FAVOUR of catch-up: on a synthetic panel with
pure bounce and ZERO stale information it returns +0.1086 (t=+13.3).


## 6. The costed books

Gross exposure exactly 1, weights formed at the close of `t` and held over `t+1` (**one** execution lag), one-way cost × NAV on turnover, borrow accrued daily on the short leg. Being self-financing at gross 1, the book's return *is* its excess-of-cash return, so the Sharpe race is already excess-vs-excess. The third book is the control: it fades the ADR's own move and never opens the home tape.

In [11]:
rows = [('catch-up  +sign(x)', R['cu_gross'], R['cu_sharpe'], R['cu_t'], R['cu_net'], R['cu_be']),
        ('residual  -sign(e)', R['re_gross'], R['re_sharpe'], R['re_t'], R['re_net'], R['re_be']),
        ('CONTROL   -sign(a)', R['rv_gross'], R['rv_sharpe'], R['rv_t'], R['rv_net'], R['rv_be']),
        ('catch-up, Japan   ', R['jp_cu_gross'], R['jp_cu_sharpe'], R['jp_cu_t'], R['jp_cu_net'], R['jp_cu_be'])]
for tag, g, s, t, n, be in rows:
    print(f'{tag}: gross {g:+6.2f}%/yr  Sharpe {s:+.3f} (t={t:+.2f})  '
          f'net(5bp/50bp) {n:+7.2f}%/yr  breakeven {be:+.2f} bps')
print(f"\nturnover {R['turnover']*100:.0f}% of NAV per day")
print('the CONTROL, which uses no home data at all, has the best gross Sharpe of the four.')

catch-up  +sign(x): gross  -0.92%/yr  Sharpe -0.049 (t=-0.23)  net(5bp/50bp)  -14.81%/yr  breakeven -0.34 bps
residual  -sign(e): gross  +2.55%/yr  Sharpe +0.161 (t=+0.72)  net(5bp/50bp)  -12.01%/yr  breakeven +0.89 bps
CONTROL   -sign(a): gross  +5.28%/yr  Sharpe +0.326 (t=+1.66)  net(5bp/50bp)   -8.57%/yr  breakeven +1.94 bps
catch-up, Japan   : gross  +2.30%/yr  Sharpe +0.090 (t=+0.41)  net(5bp/50bp)  -11.52%/yr  breakeven +0.85 bps

turnover 108% of NAV per day
the CONTROL, which uses no home data at all, has the best gross Sharpe of the four.


In [12]:
print('bootstrap Sharpe CIs (2000 draws, 21-day blocks) — all span zero:')
print(f"  catch-up: [{R['ci_cu_lo']:+.3f}, {R['ci_cu_hi']:+.3f}]  ({R['ci_cu_neg']:.1f}% of draws < 0)")
print(f"  residual: [{R['ci_re_lo']:+.3f}, {R['ci_re_hi']:+.3f}]")
print(f"  control : [{R['ci_rv_lo']:+.3f}, {R['ci_rv_hi']:+.3f}]")
print()
print('cost sweep, catch-up book (borrow held at 50 bps):')
for c, ann, sh in R['cost_grid']:
    print(f'  {c:5.1f} bps: {ann:+7.2f}%/yr  Sharpe {sh:+.3f}')
print(f"\nborrow 0 -> 300 bps moves the net from {R['borrow_lo']:+.2f}%/yr to "
      f"{R['borrow_hi']:+.2f}%/yr — the ASSUMPTION does not drive the verdict.")
print(f"variants: dollar-neutral Sharpe {R['cu_dn_sharpe']:+.3f}, "
      f"signal-weighted {R['cu_lin_sharpe']:+.3f} — no rescue.")

bootstrap Sharpe CIs (2000 draws, 21-day blocks) — all span zero:
  catch-up: [-0.469, +0.396]  (57.4% of draws < 0)
  residual: [-0.290, +0.600]
  control : [-0.080, +0.706]

cost sweep, catch-up book (borrow held at 50 bps):
    0.0 bps:   -1.16%/yr  Sharpe -0.062
    1.0 bps:   -3.89%/yr  Sharpe -0.206
    2.0 bps:   -6.62%/yr  Sharpe -0.351
    5.0 bps:  -14.81%/yr  Sharpe -0.785
   10.0 bps:  -28.46%/yr  Sharpe -1.506
   25.0 bps:  -69.42%/yr  Sharpe -3.641

borrow 0 -> 300 bps moves the net from -14.57%/yr to -16.01%/yr — the ASSUMPTION does not drive the verdict.
variants: dollar-neutral Sharpe -0.369, signal-weighted +0.010 — no rescue.


## 7. Cross-checks and the era cut

In [13]:
print(f"era cut, whole panel: 2004-2014 b1={R['era_e_b1']:+.4f} (t={R['era_e_t']:+.2f}), "
      f"2015-2026 b1={R['era_l_b1']:+.4f} (t={R['era_l_t']:+.2f})")
print(f"NVO home-index PROXY check (2016-12 on): real OMXC25/DKK b1={R['nvo_real_b1']:+.4f} "
      f"(t={R['nvo_real_t']:+.2f}) vs GDAXI/EUR proxy {R['nvo_proxy_b1']:+.4f} "
      f"(t={R['nvo_proxy_t']:+.2f}) — same null")
print(f"context: EW ADR basket excess-of-^IRX {R['ew_ann']:+.2f}%/yr Sharpe "
      f"{R['ew_sharpe']:+.3f} (t={R['ew_t']:+.2f}); on BIL cash, Sharpe "
      f"{R['ew_bil_sharpe']:+.3f} (t={R['ew_bil_t']:+.2f})")

era cut, whole panel: 2004-2014 b1=+0.0285 (t=+1.12), 2015-2026 b1=+0.0103 (t=+0.44)
NVO home-index PROXY check (2016-12 on): real OMXC25/DKK b1=+0.0194 (t=+0.43) vs GDAXI/EUR proxy +0.0381 (t=+0.78) — same null
context: EW ADR basket excess-of-^IRX +11.19%/yr Sharpe +0.515 (t=+2.67); on BIL cash, Sharpe +0.450 (t=+2.15)


## 8. Live synthetic control — the machinery, and the confound

*The cell below is the **synthetic** control — a simulated panel with a known planted answer, run live so you can see the machinery is unbiased. It is not the real tape and carries no part of the verdict.*

Two things are planted independently: a genuine catch-up lag (a share of the home loading arrives a day late) and pure bid-ask bounce (a transient pricing error that unwinds tomorrow). The point is that they are *not* interchangeable.

In [14]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from adr_catchup import data, strategy as st
import numpy as np
pl = st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=955)[0])
print(f"planted lag : b1 {pl['beta_lag']:+.4f} (t {pl['t_lag']:+.1f})  "
      f"x|a {pl['coef_x_ctrl']:+.4f} (t {pl['t_x_ctrl']:+.1f})  book Sharpe {pl['gross_sharpe']:+.2f}")
nl = [st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=955+s)[0])
      for s in range(5)]
b = np.array([d['beta_lag'] for d in nl]); tb = np.array([d['t_lag'] for d in nl])
print(f"null x5     : b1 mean {b.mean():+.4f} (sd {b.std(ddof=1):.4f}), |t|>=2 on {(abs(tb)>=2).sum()}/5")
bo = st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, bounce_vol=0.006,
                                             seed=955)[0])
print(f"pure BOUNCE, zero catch-up: residual rule fires (FM slope {bo['fm_slope_e']:+.4f}, "
      f"t {bo['fm_t_e']:+.1f}) but b1 stays silent ({bo['beta_lag']:+.4f}, t {bo['t_lag']:+.2f})")

planted lag : b1 +0.2510 (t +47.8)  x|a +0.2490 (t +40.8)  book Sharpe +7.54


null x5     : b1 mean -0.0006 (sd 0.0046), |t|>=2 on 0/5


pure BOUNCE, zero catch-up: residual rule fires (FM slope -0.1535, t -20.6) but b1 stays silent (+0.0060, t +0.96)


> 💡 **In plain words.** The last line is why the residual test never carries the verdict: a market with *no* stale information at all, only a jumpy bid-ask spread, makes it look like ADRs are catching up. The lagged-home-move test is immune to that, because yesterday's foreign index is independent of today's pricing error.

## Verdict

- **Signal — Weak.** Pooled over eight ADRs the lagged home loading is +0.0209 (HAC *t* = +1.15), and decomposed it is entirely the FX leg (+0.0683, *t* = +2.79) against a dead index leg (+0.0023, *t* = +0.12) — a snapshot artefact in a market that never closes. Japan is the exception the clock predicts: b1 = **+0.0809** (*t* = **+3.25**), index leg +0.0620 (*t* = +2.43), both names alone (*t* = +3.08 / +2.75), positive in all four blocks and both halves (+2.60 / +2.25) — and the UK, whose home markets close mid-session, is -0.0479 (*t* = -1.80), the mechanism's own prediction confirmed. What stops that being a result (§3½): deleting the 48 largest lagged home moves — 0.5% of the rows — takes it to +0.0169 (*t* = +0.67), where the identical trim moves a synthetic *linear* planted lag by 1%; winsorized it is +0.0603 (*t* = +2.62), and the bucket sort pays +1.60 bp Q5 − Q1 with no gradient. On top of that the bettable γ is +0.0331 (*t* = +1.43) in Japan and -0.0077 overall, clearing |*t*| = 2 nowhere; the home tape adds +0.0056 (*t* = +0.27) once the ADR's own move is controlled for, against a test that bounce biases *in its favour*; and eight surviving mega-caps are a survivor-picked universe.
- **Tradability — Mirage.** Catch-up book -0.92%/yr gross (Sharpe -0.049, *t* = -0.23) with a **negative** breakeven cost; Japan-only breaks even at 0.85 bps one-way; 108% daily turnover turns 5 bps into -14.81%/yr. Every bootstrap CI spans zero, no weighting variant rescues it, and the no-home-data control beats it gross (+0.326 vs -0.049). Statistically visible, economically buried.